# Lab 8: Find and repair a data alignment defect

**Goal:** Check applied action timing, replay one saved episode, and repair a labelled synthetic fault. Then collect 5–10 short recovery demonstrations with the keyboard tool and assess one bounded ACT continuation on held-out development states.

**Inputs:** Tracked source-action audit and synthetic recording fixture. The optional live keyboard path uses local LIBERO assets. **Compute:** CPU for the saved analysis; local GPU for the continuation. Start Jupyter from the repository root.

## 1. Inspect source actions without inventing missing time

The audit reads all 39 bowl-plate training episodes and all 3,661 source actions. Near-zero means that the largest absolute value among the six arm action components is at or below a declared threshold. The gripper is excluded. A low count can be consistent with filtering, but the source filtering history is unknown. Dataset timestamps do not recover any removed physical time.

In [ ]:
from pathlib import Path
import hashlib
import json
import matplotlib.pyplot as plt
import numpy as np
from vla_lab.data.recording import RecordedEpisode, replay_episode, repair_synthetic_timing_defect

WORKSPACE = Path.cwd().resolve()
audit = json.loads((WORKSPACE / "artifacts/reference/data_quality/source_action_audit.json").read_text())
assert audit["training_episode_count"] == 39 and audit["training_frame_count"] == 3661
print("Episode lengths:", audit["episode_length_min"], "to", audit["episode_length_max"])
for threshold, count in audit["near_zero_counts"].items():
    print("Arm threshold", threshold, "count", count, "fraction", round(count / 3661, 4))
print(audit["timing_limit"])

## 2. Check a saved recording and its fault label

This small fixture came from a deterministic synthetic simulator. It is not a LIBERO demonstration. It has 12 applied actions and 13 observations at 50 ms control ticks. The defect changes action tick labels by one tick; it does not change applied action values. The repaired reference restores the clean labels.

In [ ]:
fixture = json.loads((WORKSPACE / "artifacts/reference/data_quality/synthetic_fixture_manifest.json").read_text())
recordings = {}
for name, item in fixture["files"].items():
    path = WORKSPACE / item["path"]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == item["sha256"]
    recordings[name] = RecordedEpisode.load(path)
clean = recordings["synthetic_clean"]
defect = recordings["synthetic_defect"]
repaired = repair_synthetic_timing_defect(defect)
assert clean.actions.shape == (12, 7) and len(clean.observations) == 13
assert np.array_equal(defect.action_ticks, clean.action_ticks + 1)
assert np.array_equal(repaired.action_ticks, clean.action_ticks)
assert np.array_equal(repaired.actions, clean.actions)
print("Clean action ticks:", clean.action_ticks.tolist())
print("Faulty labels:", defect.action_ticks.tolist())
print("Fault source:", defect.defect["source_history"])

In [ ]:
fig, axis = plt.subplots(figsize=(8, 3))
axis.step(clean.action_ticks, clean.actions[:, 0], where="post", label="clean x action")
axis.step(defect.action_ticks, defect.actions[:, 0], where="post", label="faulty x label", linestyle="--")
axis.set(xlabel="labelled control tick", ylabel="relative x action", title="One-tick label error on unchanged actions")
axis.legend()
plt.show()

## 3. Replay the saved actions

Replay starts from the saved reset seed and applies the saved actions in order. The fixture checks the numeric robot state after each action. A real LIBERO replay can differ if its reset or simulator version changes; the saved file also stores its camera observations for inspection.

In [ ]:
class FixtureReplay:
    def reset(self, seed=None):
        self.state = np.zeros(8, dtype=np.float32)
        return {"agent_pos": self.state.copy()}, {}
    def step(self, action):
        self.state[:6] += action[:6]
        self.state[6] = action[6]
        return {"agent_pos": self.state.copy()}, 0.0, False, False, {}

replayed = replay_episode(FixtureReplay(), clean)
assert len(replayed) == len(clean.observations)
assert all(np.allclose(a["agent_pos"], b["agent_pos"]) for a, b in zip(replayed, clean.observations, strict=True))
print("Replay matched all", len(replayed), "saved numeric states.")

## 4. Check the ACT recovery window

The recorded controller runs at 20 Hz, with one action every 50 ms. The adapter keeps every recorded observation and applied action at that tick. It does not resample to match the source dataset's 10 Hz timestamp labels. It keeps each action window inside one episode and marks the padded tail.

In [ ]:
from vla_lab.data.recovery_dataset import RecoveryDataset
fixture_dataset = RecoveryDataset(
    [WORKSPACE / fixture["files"]["synthetic_clean"]["path"]],
    task_id="synthetic_alignment_fixture", expected_reset_seeds={7}
)
first = fixture_dataset[0]
last = fixture_dataset[len(fixture_dataset) - 1]
assert len(fixture_dataset) == 12
assert first["observation.images.image"].shape == (3, 256, 256)
assert first["observation.state"].shape == (8,)
assert first["action"].shape == (20, 7)
assert last["action_is_pad"].sum().item() == 19
print("20 Hz controller samples:", len(fixture_dataset), "padded values in last window:", int(last["action_is_pad"].sum()))

## 5. Record short recovery demonstrations

The keyboard tool uses the same seven-value relative action schema as policy rollout. It records the action sent to the simulator with the observation at that tick. It allows only development states. Run this command in a desktop terminal with the pinned LIBERO assets:

```bash
.venv/bin/python -m vla_lab.data.keyboard --task bowl_plate --state 0 --output .local-data/keyboard-recordings/my-first-demo.npz
```

Press X to save or Q to discard. Record 5–10 short episodes. Keep the state IDs you record out of the later development assessment. The reference continuation uses a separately labelled supplied set, so the notebook can run without human input.

In [ ]:
saved = sorted((WORKSPACE / ".local-data/keyboard-recordings").glob("*.npz"))
print("Local keyboard recordings:", len(saved))
for path in saved[:10]:
    episode = RecordedEpisode.load(path)
    print(path.name, "actions", len(episode.actions), "task", episode.task_id)

## 6. Read the bounded recovery reference

The supplied reference has five successful ACT-generated recordings from development states 0–4. It is not human data. The checked ACT continuation used 200 new optimizer steps and kept every recorded 50 ms controller tick. Assessment used development states 5–19 only. Read the saved attempts and compare them with the unchanged base checkpoint. An earlier adapter kept only even ticks to match dataset timestamp labels; that diagnostic run failed all 15 assessment states and is not the reference recipe.

In [ ]:
teacher = json.loads((WORKSPACE / 'artifacts/reference/data_quality/teacher_manifest.json').read_text())
reference = json.loads((WORKSPACE / 'artifacts/reference/data_quality/recovery_reference.json').read_text())
assert teacher['source_type'] == reference['source_type'] == 'policy_generated'
assert teacher['attempted_state_ids'] == list(range(5))
assert reference['assessment_state_ids'] == list(range(5, 20))
assert len(teacher['recordings']) == reference['accepted_recordings'] == 5
local_paths = [WORKSPACE / item['path'] for item in teacher['recordings']]
if all(path.exists() for path in local_paths):
    for item, path in zip(teacher['recordings'], local_paths, strict=True):
        assert hashlib.sha256(path.read_bytes()).hexdigest() == item['sha256']
        episode = RecordedEpisode.load(path)
        assert episode.reset_seed == item['reset_seed']
        assert len(episode.observations) == len(episode.actions) + 1
    real_dataset = RecoveryDataset(
        local_paths, task_id='bowl_plate',
        expected_reset_seeds={item['reset_seed'] for item in teacher['recordings']},
    )
    assert len(real_dataset) == reference['tick_aligned_sample_count'] == 375
    print('Checked local ACT-generated recordings:', len(local_paths), 'controller samples:', len(real_dataset))
else:
    print('The full local recordings are absent. The tracked manifest and assessment remain available.')

In [ ]:
heldout_rows = {}
for name in ('base', 'tick_aligned', 'resampled_diagnostic'):
    item = reference['results'][name]
    path = WORKSPACE / item['episodes_file']
    rows = [json.loads(line) for line in path.read_text().splitlines()]
    assert [row['state_id'] for row in rows] == reference['assessment_state_ids']
    assert all(row['reset_seed'] == 20260930 + row['state_id'] for row in rows)
    assert sum(row['success'] for row in rows) == item['successes']
    heldout_rows[name] = rows
    print(name, f"{item['successes']}/15", 'failed states:', item['failed_state_ids'])
assert reference['paired_base_only_state_ids'] == [5]
assert reference['paired_continuation_only_state_ids'] == [8]
print('The corrected continuation changed one failure state but did not increase total success.')
print('The resampled run is a timing-defect diagnostic, not a valid recovery recipe.')

## Completion check

Save the [exercise](exercise.md) before you read the [solution guide](../../solutions/08-data-quality.md). Separate the synthetic fault from an observed source-data property. Do not infer removed physical time from dataset timestamps.